# Week 6 – Interactive and Narrative Visualization

In [14]:
from pathlib import Path

# Get current notebook directory
current_dir = Path().resolve()

# Build dataset path
data_path = current_dir.parent.parent / "Data" / "2003_2026.csv"

data_path

PosixPath('/Users/katherinakronborg/Desktop/Data Analysis 2026/social_data_viz_2026_group_75/Data/2003_2026.csv')

## Exercise 1.1 – Explanatory Data Visualization

### 1. Three key elements when designing an explanatory visualization

The video highlights three important principles when designing explanatory visualizations:

1. **Start with a clear question or message**  
   The visualization should communicate a specific insight or takeaway from the data.

2. **Allow exploration**  
   Interactive elements help users engage with the data and discover patterns themselves.

3. **Know your audience**  
   Visualizations should be designed based on the knowledge level and expectations of the intended audience.

---

### 2. Example visualization following the principles

Example: **Global COVID-19 Dashboard by Johns Hopkins University**

https://coronavirus.jhu.edu/map.html

This interactive dashboard follows the principles *overview first, zoom and filter, details on demand*.

**Overview first**

The dashboard initially presents a global map showing the overall spread of COVID-19 cases across countries. This high-level view allows users to quickly understand the global situation.

**Zoom and filter**

Users can zoom into specific regions or countries and filter the data to examine different metrics such as confirmed cases, deaths, or recoveries. This allows deeper exploration of specific areas.

**Details on demand**

When hovering over a country or region, additional information such as the exact number of cases and deaths appears. This allows users to access detailed data without cluttering the main visualization.

*(A screenshot of the dashboard can be added here to illustrate the interaction.)*

---

### 3. Difference between explanatory and exploratory data analysis

**Exploratory data analysis (EDA)** is used by analysts to investigate a dataset, discover patterns, and generate insights. It focuses on understanding the data and often involves many quick and flexible visualizations.

**Explanatory data analysis** focuses on communicating insights that have already been discovered. The goal is to clearly present the key findings to an audience in a way that is easy to understand.

## Part 2: Interactive Visualizations with Plotly

In this section we create an **interactive visualization** of how personal focus crimes are distributed across the 24 hours of the day.

Unlike the static visualizations from earlier weeks, Plotly allows the user to interact with the data through:

- Hover tooltips
- Zooming and panning
- Clicking legend items to show or hide crime categories

The goal is to make it easier to compare **temporal patterns across crime types**. By toggling crime types in the legend, users can isolate individual distributions or directly compare multiple crimes.

The data is normalized so that each crime type forms a **probability distribution across the 24 hours of the day**. This allows us to compare the *shape* of crime patterns rather than their total counts.

### Exercise 2.1 – Interactive Hourly Crime Patterns

In this exercise we create an interactive bar chart showing how each personal focus crime is distributed across the 24 hours of the day.

The counts are **normalized within each crime category**, meaning the bars represent the probability that a given crime occurs at a specific hour.

The visualization is interactive, allowing the user to toggle crime categories on and off in the legend to compare their hourly patterns.

In [15]:
# Focus crimes from Week 2
focus_crimes = [
    'Robbery',
    'Motor Vehicle Theft',
    'Weapons Offense',
    'Rape',
    'Disorderly Conduct',
    'Larceny Theft'
]

# Filter dataset to only those crimes
df_focus = df[df["Incident Category"].isin(focus_crimes)]

df_focus.head()

,Incident Category,Incident Datetime,Incident Day of Week,Incident Code,Incident Description,Police District,Longitude,Latitude,Hour,Year
0,Robbery,2004-11-22 17:50:00,Monday,3074,"ROBBERY, BODILY FORCE",Ingleside,-122.420084,37.708311,17,2004
1,Motor Vehicle Theft,2005-10-18 20:00:00,Tuesday,7021,STOLEN AUTOMOBILE,Park,-120.500000,90.000000,20,2005
2,Motor Vehicle Theft,2004-02-15 02:00:00,Sunday,7021,STOLEN AUTOMOBILE,Southern,-120.500000,90.000000,2,2004
7,Motor Vehicle Theft,2016-03-03 19:30:00,Thursday,7020,STOLEN AND RECOVERED VEHICLE,Taraval,-122.463545,37.707968,19,2016
11,Larceny Theft,2012-12-21 19:15:00,Friday,6244,GRAND THEFT FROM LOCKED AUTO,Central,-122.406832,37.796903,19,2012


In [16]:
# Group data by hour and crime type
hourly_counts = (
    df_focus
    .groupby(["Hour", "Incident Category"])
    .size()
    .reset_index(name="Count")
)

# Normalize counts within each crime category
hourly_counts["NormalizedCount"] = (
    hourly_counts
    .groupby("Incident Category")["Count"]
    .transform(lambda x: x / x.sum())
)

hourly_counts.head()

,Hour,Incident Category,Count,NormalizedCount
0,0,Disorderly Conduct,1502,0.048588
1,0,Larceny Theft,33549,0.043500
2,0,Motor Vehicle Theft,7146,0.039452
3,0,Rape,918,0.101887
4,0,Robbery,3893,0.050591


In [18]:
import plotly.express as px

fig = px.bar(
    hourly_counts,
    x="Hour",
    y="NormalizedCount",
    color="Incident Category",
    barmode="overlay",
    opacity=0.6,
    title="Normalized Hourly Distribution of Personal Focus Crimes (2003–2026)",
)

# Set axis ranges explicitly
fig.update_layout(
    xaxis=dict(title="Hour of Day", range=[-0.5, 23.5]),
    yaxis=dict(title="Normalized Crime Frequency", range=[0, 0.12]),
    legend=dict(
        title="Crime Type",
        x=1.02,
        y=1,
        bgcolor="rgba(0,0,0,0)"
    )
)

# Start with all traces hidden
for trace in fig.data:
    trace.visible = "legendonly"

fig.show()

### Interpretation of the Interactive Visualization

The interactive chart reveals several interesting temporal patterns across the personal focus crimes.

**Crime types with similar hourly patterns**

Some crimes follow similar patterns throughout the day. For example, **Larceny Theft** and **Motor Vehicle Theft** tend to increase during the afternoon and evening hours. These crimes peak roughly between the late afternoon and early night, which may reflect times when more people are outside, commuting, or leaving vehicles unattended.

**Crime types with different patterns**

Other crimes show very different hourly distributions. **Disorderly Conduct** appears to peak more during late night and early morning hours, while crimes such as **Weapons Offense** are more evenly distributed throughout the day. These differences highlight how certain crimes are associated with different daily activities or social environments.

**Comparison with Week 2**

Compared to the static visualization from Week 2, the overall patterns appear similar. However, the interactive visualization makes it easier to isolate individual crime types and examine their hourly distributions without interference from other categories.

**Effect of interactivity**

The interactive features significantly improve the exploration of the data. Clicking on crime types in the legend allows individual categories to be toggled on or off. When double-clicking a crime type, the chart isolates that category and the y-axis automatically rescales, making the pattern clearer and easier to interpret. Hovering over bars also reveals the exact normalized value and hour, which provides more detailed information than the static plots from Week 2.

### Exercise 2.2 – Animated Crime Patterns

In this exercise we extend the analysis of hourly crime distributions by introducing **animation over time**.

Instead of calculating one overall hourly distribution, we compute **separate normalized hourly distributions for each year**. This allows us to observe how crime patterns throughout the day may have changed over time.

We then create an **animated line chart using Plotly**, where each frame represents a different year. The animation allows us to explore how the temporal distribution of crime evolves across years.

To ensure the visualization remains comparable across frames, we fix the y-axis range so that Plotly does not automatically rescale the axis for each year.

In [19]:
# Group by year, hour, and crime type
yearly_hourly = (
    df_focus
    .groupby(["Year", "Hour", "Incident Category"])
    .size()
    .reset_index(name="Count")
)

# Normalize within each year AND crime type
yearly_hourly["NormalizedCount"] = (
    yearly_hourly
    .groupby(["Year", "Incident Category"])["Count"]
    .transform(lambda x: x / x.sum())
)

yearly_hourly.head()

,Year,Hour,Incident Category,Count,NormalizedCount
0,2003,0,Disorderly Conduct,58,0.055449
1,2003,0,Larceny Theft,891,0.034027
2,2003,0,Motor Vehicle Theft,628,0.041110
3,2003,0,Rape,47,0.105381
4,2003,0,Robbery,177,0.056858


In [22]:
df_focus.groupby(["Year", "Incident Category"]).size().unstack()

Incident Category,Disorderly Conduct,Larceny Theft,Motor Vehicle Theft,Rape,Robbery,Weapons Offense
Year,,,,,,
2003,1046,26185,15276,446,3113,1196
2004,1042,24337,17816,403,3297,1174
2005,946,25226,18103,423,3490,1289
2006,601,27227,7263,402,4031,1246
2007,997,25599,6444,431,3928,1245
2008,1219,25636,6042,485,4150,1357
2009,1074,25419,5170,485,3508,1367
2010,1000,24212,4336,553,3232,1243
2011,875,25629,4743,560,3218,1184


In [20]:
fig = px.line(
    yearly_hourly,
    x="Hour",
    y="NormalizedCount",
    color="Incident Category",
    animation_frame="Year",
    title="Hourly Distribution of Personal Focus Crimes by Year (2003–2026)",
    range_y=[0, 0.2]
)

# Start with traces hidden so users can toggle them
fig.update_traces(visible="legendonly")

fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Normalized Crime Frequency",
    legend_title="Crime Type"
)

fig.show()

### Interpretation of the Animated Visualization

Watching the animation shows that the **overall shapes of the hourly distributions remain relatively stable across the years**. Most crime types continue to follow similar daily patterns throughout the dataset. For example, crimes such as **Motor Vehicle Theft and Larceny Theft** tend to increase during the afternoon and evening hours, while **Disorderly Conduct** shows relatively higher activity during late night and early morning hours.

Some crime types show **small shifts in intensity at certain hours**, but the general structure of the distributions does not drastically change over time. This suggests that the daily rhythms of these crimes are relatively persistent.

**Effect of COVID (2020–2021)**  
During the COVID years, the patterns appear slightly smoother and somewhat shifted toward later hours in the day for some crimes. This may reflect changes in daily routines during lockdowns and reduced nighttime activity in public spaces.

**Noisier patterns in recent years**  
Some crime types, particularly **Rape and Weapons Offense**, appear noisier in the more recent years. This likely occurs because the **total number of incidents for these crimes is relatively low**, so when the counts are normalized small changes in counts can produce larger fluctuations in the distribution.

### Line Chart vs Bar Chart

When comparing the animated **line chart** and **bar chart**, the **line chart makes the temporal evolution easier to see**. The lines highlight the overall shape of the hourly distributions and make it easier to track how patterns shift between hours.

The animated bar chart is more cluttered because each hour contains multiple bars for different crime types. This makes it harder to visually follow the changes across years, especially when many categories are displayed simultaneously.

### Reflection on Animation

Animation is helpful for observing how patterns evolve over time, but it also has limitations. Because each year is shown sequentially, it can be difficult to compare multiple years directly.

An alternative approach such as **small multiples (one subplot per year)** could make comparisons easier because all years would be visible at once. However, animation provides a more engaging and intuitive way to explore how patterns change gradually over time.

Overall, animation works well for **exploratory analysis**, while small multiples may be better for **clear comparisons across years**.

### Part 3.1 – Narrative Visualization

**1. What is the Oxford English Dictionary's definition of a narrative?**

According to the paper, the *Oxford English Dictionary* defines a narrative as:

> “An account of a series of events, facts, etc., given in order and with the establishing of connections between them.” 

This definition highlights that a narrative is not just a list of events or facts, but a structured sequence where relationships and connections between events are clearly established. In the context of data visualization, this means guiding the viewer through data in a meaningful order so that the underlying story becomes understandable.

---

**2. What is your favorite visualization among the examples in Section 3? Explain why in a few words.**

Our favorite example is the **Gapminder Human Development Trends** visualization.

We like this example because it clearly explains complex global development data through step-by-step storytelling. The visualization gradually introduces information using animations, annotations, and transitions, which helps the viewer understand patterns such as changes in poverty and health over time. It also allows some interactivity after the narrative explanation, making it easier to explore the data further while still understanding the main story first. 


### Part 3.2 – Connecting the Dots

**Author-driven vs Reader-driven visualizations**

The interactive visualizations created in Exercises 2.1 and 2.2 are mostly **reader-driven**. They allow the viewer to explore the data by interacting with the plot, choosing which crime types to view, and moving through time in the animated visualization. This means the viewer determines what patterns to focus on rather than being guided through a fixed narrative.

To move these visualizations further toward the **author-driven** end of the spectrum, we could add stronger narrative structure. For example, we could include annotations highlighting important patterns, introduce the visualization with explanatory text, or guide the viewer step-by-step through specific findings. This could follow a structure similar to the **martini glass model**, where the author first explains key insights before allowing the viewer to freely explore the data. 

---

**Designing visualizations for different audiences**

For the **San Francisco Board of Supervisors**, an **interactive slideshow or animated visualization** would work well. This type of narrative visualization would allow us to guide the audience through the most important findings while still allowing limited interaction. For example, we could first show how crime patterns change over the day, then highlight specific crime types that peak at certain hours, and finally show how these patterns have changed over time. This structured narrative approach would help policymakers quickly understand the key insights without needing to explore the data themselves.

For a **peer-reviewed journal**, the visualization would need to be **static**, since it would appear as a figure in a PDF. In this case, an **annotated chart or partitioned poster style visualization** would be more appropriate. Multiple small charts could show different crime types or years side-by-side, with annotations highlighting important trends. This would allow the reader to understand the findings without relying on animation or interactivity.

---

**Assumptions and feedback loops**

If a city official used an animated visualization to argue for increased late-night police patrols in certain neighborhoods, they would be making several assumptions. One assumption would be that higher reported crime at certain hours directly reflects actual crime activity rather than differences in reporting or policing practices. Another assumption is that increasing patrols would reduce crime rather than simply increasing the number of recorded incidents.

This connects to the **feedback loops discussed in the Richardson et al. reading from Week 1**. Increased policing in specific areas may lead to more reported crimes simply because officers are present to detect and record them. These new data points could then be used to justify even more policing in those same areas, reinforcing the original assumption. As a result, the visualization could unintentionally contribute to self-reinforcing policy decisions if the broader context behind the data is not carefully considered.


## Further Additional Exercises

To further explore interactive data visualization techniques, we experimented with additional Plotly chart types and dashboard features. These exercises help demonstrate how different visualization styles can highlight different aspects of the crime dataset and support both exploratory analysis and presentation.

In particular, we explored hierarchical charts (such as sunburst and treemap), interactive dashboard controls (dropdown menus), and small-multiple visualizations to compare patterns across years.

In [24]:
import plotly.express as px

# Aggregate counts
sunburst_df = (
    df_focus
    .groupby(["Incident Category", "Hour"])
    .size()
    .reset_index(name="Count")
)

fig = px.sunburst(
    sunburst_df,
    path=["Incident Category", "Hour"],
    values="Count",
    title="Sunburst Chart:Crime Distribution by Category and Hour"
)

fig.show()

In [25]:
treemap_df = (
    df_focus
    .groupby(["Incident Category", "Hour"])
    .size()
    .reset_index(name="Count")
)

fig = px.treemap(
    treemap_df,
    path=["Incident Category", "Hour"],
    values="Count",
    title="Treemap of Crime Categories by Hour"
)

fig.show()

In [26]:
import plotly.graph_objects as go

years = sorted(df_focus["Year"].unique())

fig = go.Figure()

for year in years:
    yearly = (
        df_focus[df_focus["Year"] == year]
        .groupby("Hour")
        .size()
        .reset_index(name="Count")
    )

    fig.add_trace(
        go.Scatter(
            x=yearly["Hour"],
            y=yearly["Count"],
            mode="lines",
            name=str(year),
            visible=(year == years[0])
        )
    )

buttons = []

for i, year in enumerate(years):
    visibility = [False] * len(years)
    visibility[i] = True

    buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Hourly Crime Distribution — {year}"}]
        )
    )

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True
        )
    ],
    title=f"Hourly Crime Distribution — {years[0]}",
    xaxis_title="Hour of Day",
    yaxis_title="Crime Count"
)

fig.show()

In [27]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

years = sorted(df_focus["Year"].unique())

fig = make_subplots(
    rows=len(years),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[str(y) for y in years]
)

for i, year in enumerate(years, start=1):
    yearly = (
        df_focus[df_focus["Year"] == year]
        .groupby("Hour")
        .size()
        .reset_index(name="Count")
    )

    fig.add_trace(
        go.Scatter(
            x=yearly["Hour"],
            y=yearly["Count"],
            mode="lines",
            name=str(year)
        ),
        row=i,
        col=1
    )

fig.update_layout(
    height=300 * len(years),
    title="Small Multiples: Hourly Crime Distribution by Year",
    xaxis_title="Hour of Day",
    yaxis_title="Crime Count"
)

fig.show()

### Comparison: Animation vs Small Multiples

The **animated visualization** is useful for exploring how crime patterns evolve over time, since it highlights changes dynamically and keeps the viewer focused on one year at a time. However, it can make it difficult to compare multiple years simultaneously.

The **small multiples visualization** shows all years at once in separate panels, which makes it easier to directly compare patterns between years. This format is often better for presentation in reports or papers where the audience needs to see multiple time periods simultaneously.

Overall, animation works well for **interactive exploration**, while small multiples are generally better for **clear comparison and presentation**.